In [2]:

%pip install transformers torch accelerate


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\AMIT\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:

import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.15.0+cpu
CUDA available: False


In [4]:

custom_texts = [
    "Artificial intelligence enables machines to perform tasks that usually require human intelligence.",
    "Machine learning allows computers to learn patterns from data.",
    "Natural language processing helps computers understand human language.",
    "Deep learning uses neural networks with multiple layers.",
    "GPT-2 is a transformer-based language model that can generate text.",
    "Text generation is the task of predicting and producing the next tokens in a sequence.",
    "Data preprocessing improves the quality of machine learning experiments.",
    "Python is widely used for artificial intelligence and data science.",
    "Language models learn statistical patterns from their training text.",
    "Artificial intelligence is being applied in education, healthcare, and agriculture.",
    "Tokenization converts text into smaller units that a model can process.",
    "A transformer uses attention mechanisms to process relationships between tokens.",
    "Machine translation converts text from one language into another.",
    "Text summarization produces a shorter version of a longer document.",
    "Chatbots use language processing techniques to respond to user questions."
]

dataset_path = "custom_dataset.txt"

with open(dataset_path, "w", encoding="utf-8") as file:
    file.write("\n".join(custom_texts))

print("Custom dataset created:", dataset_path)
print("Number of text samples:", len(custom_texts))


Custom dataset created: custom_dataset.txt
Number of text samples: 15


In [5]:

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")


Tokenizer loaded successfully!


In [6]:

tokenized_samples = tokenizer(
    custom_texts,
    truncation=True,
    max_length=128,
    padding="max_length"
)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        return {
            "input_ids": torch.tensor(
                self.encodings["input_ids"][index],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                self.encodings["attention_mask"][index],
                dtype=torch.long
            )
        }

train_dataset = TextDataset(tokenized_samples)

print("Tokenization completed!")
print("Training samples:", len(train_dataset))


Tokenization completed!
Training samples: 15


In [7]:

model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

print("GPT-2 model loaded successfully!")
print("Parameters:", sum(p.numel() for p in model.parameters()))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 model loaded successfully!
Parameters: 124439808


C:\Users\AMIT\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AMIT\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [9]:

import transformers
print("Transformers version:", transformers.__version__)


Transformers version: 5.21.0


In [10]:

training_args = TrainingArguments(
    output_dir="./gpt2_custom_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Training configuration ready!")


Training configuration ready!


In [11]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

train_result = trainer.train()

print("Training completed!")
print("Training loss:", train_result.training_loss)


C:\Users\AMIT\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.260765
10,3.204475
15,2.567726
20,2.370497


Training completed!
Training loss: 2.975107749303182


In [12]:

save_path = "./gpt2_custom_finetuned"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved at:", save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at: ./gpt2_custom_finetuned


In [13]:

def generate_text(prompt, max_new_tokens=60):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.95,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

prompt = "Artificial intelligence"
generated_text = generate_text(prompt)

print("Prompt:", prompt)
print("\nGenerated text:\n")
print(generated_text)


Prompt: Artificial intelligence

Generated text:

Artificial intelligence is the technology that works by learning from information. It can learn about a person's personality and patterns, making decisions based on their emotion rather than just thoughts or behaviors.
 "When we do something with our human brains, it makes sense for us to consider more of what other people have been thinking


In [14]:

test_prompts = [
    "Machine learning",
    "Natural language processing",
    "GPT-2 is a language model"
]

for prompt in test_prompts:
    print("=" * 60)
    print("Prompt:", prompt)
    print("Generated:", generate_text(prompt, max_new_tokens=40))


Prompt: Machine learning
Generated: Machine learning is the process of finding patterns. Computers learn by taking inputs from groups and then processing them into neural networks using a set algorithm that can predict information about those regions within their context, such as when
Prompt: Natural language processing
Generated: Natural language processing techniques provide a more flexible, accurate and reliable way to process large numbers of text. A basic model that combines the information from multiple sources can form an algorithm suitable for analyzing human-computer interactions in complex
Prompt: GPT-2 is a language model
Generated: GPT-2 is a language model that uses machine learning and network theory to improve the efficiency of large numbers. It was originally developed by NLP research group at Stanford University in 2006, which has been used for many field experiments like gen


In [15]:

print("Saved model files:")
for filename in sorted(os.listdir(save_path)):
    print("-", filename)


Saved model files:
- config.json
- generation_config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin
